<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/04_tfrecord_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04 — TFRecord Generation

This notebook converts the validated normalized manifests into TensorFlow Object Detection API TFRecords.

Pipeline:

```text
Validated JSONL manifest
        ↓
tf.train.Example
        ↓
train.record
validation.record
test.record
```

The notebook does not modify source images or annotations.

In [4]:
from pathlib import Path
import json
import os

import tensorflow as tf
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

ANNOTATION_DIR = PROCESSED_DIR / "annotations"
SPLIT_DIR = ANNOTATION_DIR / "splits"

TFRECORD_DIR = PROCESSED_DIR / "tfrecords"
TFRECORD_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP_PATH = PROJECT_DIR / "configs" / "label_map.pbtxt"

TARGET_CLASSES = {
    1: "Crack",
    2: "Pothole",
    3: "Surface deterioration",
}

print("TensorFlow:", tf.__version__)
print("Label map:", LABEL_MAP_PATH)

TensorFlow: 2.21.0
Label map: /content/drive/MyDrive/CS PROJECT II/configs/label_map.pbtxt


In [5]:
# ---------------------------------------------------------
# Notebook 04 preflight checks
# ---------------------------------------------------------

TRAIN_MANIFEST = SPLIT_DIR / "train.jsonl"
VALIDATION_MANIFEST = SPLIT_DIR / "validation.jsonl"
TEST_MANIFEST = SPLIT_DIR / "test.jsonl"

print("Processed annotation directory:", ANNOTATION_DIR)
print("Split directory:", SPLIT_DIR)

print("\nRequired split files:")
print("train.jsonl:", TRAIN_MANIFEST.exists())
print("validation.jsonl:", VALIDATION_MANIFEST.exists())
print("test.jsonl:", TEST_MANIFEST.exists())

print("\nLabel map:")
print(LABEL_MAP_PATH)
print("Exists:", LABEL_MAP_PATH.exists())

assert TRAIN_MANIFEST.exists(), "train.jsonl not found"
assert VALIDATION_MANIFEST.exists(), "validation.jsonl not found"
assert TEST_MANIFEST.exists(), "test.jsonl not found"

print("\nTFRecord preflight passed.")

Processed annotation directory: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/annotations
Split directory: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/annotations/splits

Required split files:
train.jsonl: True
validation.jsonl: True
test.jsonl: True

Label map:
/content/drive/MyDrive/CS PROJECT II/configs/label_map.pbtxt
Exists: True

TFRecord preflight passed.


In [3]:
# ---------------------------------------------------------
# Create label_map.pbtxt if missing
# ---------------------------------------------------------

LABEL_MAP_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

label_map_text = """item {
  id: 1
  name: 'Crack'
}

item {
  id: 2
  name: 'Pothole'
}

item {
  id: 3
  name: 'Surface deterioration'
}
"""

LABEL_MAP_PATH.write_text(
    label_map_text,
    encoding="utf-8"
)

print("Label map created:")
print(LABEL_MAP_PATH)

print("\nContents:")
print(
    LABEL_MAP_PATH.read_text(
        encoding="utf-8"
    )
)

Label map created:
/content/drive/MyDrive/CS PROJECT II/configs/label_map.pbtxt

Contents:
item {
  id: 1
  name: 'Crack'
}

item {
  id: 2
  name: 'Pothole'
}

item {
  id: 3
  name: 'Surface deterioration'
}



## 1. Verify the Label Map

Do not overwrite the repository label map here. The training pipeline and backend should use the same class IDs.

In [6]:
print(LABEL_MAP_PATH.read_text(encoding="utf-8"))

for class_id, name in TARGET_CLASSES.items():
    print(f"Expected: {class_id} -> {name}")

item {
  id: 1
  name: 'Crack'
}

item {
  id: 2
  name: 'Pothole'
}

item {
  id: 3
  name: 'Surface deterioration'
}

Expected: 1 -> Crack
Expected: 2 -> Pothole
Expected: 3 -> Surface deterioration


## 2. TF Example Helpers

In [7]:
def bytes_feature(value):
    if isinstance(value, str):
        value = value.encode("utf-8")
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def float_list_feature(values):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[float(v) for v in values]))

def bytes_list_feature(values):
    encoded = [
        v.encode("utf-8") if isinstance(v, str) else v
        for v in values
    ]
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=encoded))

def create_tf_example(record):
    image_path = Path(record["image_path"])

    with tf.io.gfile.GFile(str(image_path), "rb") as fid:
        encoded = fid.read()

    with Image.open(image_path) as img:
        width, height = img.size
        filename = image_path.name
        image_format = image_path.suffix.lower().replace(".", "").encode("utf-8")

    xmins, xmaxs, ymins, ymaxs = [], [], [], []
    class_names, class_ids = [], []

    for obj in record["objects"]:
        class_name = obj["class_name"]
        class_id = next(
            cid for cid, name in TARGET_CLASSES.items()
            if name == class_name
        )

        xmins.append(obj["xmin"])
        xmaxs.append(obj["xmax"])
        ymins.append(obj["ymin"])
        ymaxs.append(obj["ymax"])
        class_names.append(class_name)
        class_ids.append(class_id)

    features = tf.train.Features(feature={
        "image/height": int64_feature(height),
        "image/width": int64_feature(width),
        "image/filename": bytes_feature(filename),
        "image/source_id": bytes_feature(filename),
        "image/encoded": bytes_feature(encoded),
        "image/format": bytes_feature(image_format),
        "image/object/bbox/xmin": float_list_feature(xmins),
        "image/object/bbox/xmax": float_list_feature(xmaxs),
        "image/object/bbox/ymin": float_list_feature(ymins),
        "image/object/bbox/ymax": float_list_feature(ymaxs),
        "image/object/class/text": bytes_list_feature(class_names),
        "image/object/class/label": tf.train.Feature(
            int64_list=tf.train.Int64List(value=class_ids)
        ),
    })

    return tf.train.Example(features=features)

## 3. Generate TFRecords

In [8]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_tfrecord(split_name):
    manifest_path = SPLIT_DIR / f"{split_name}.jsonl"
    output_path = TFRECORD_DIR / f"{split_name}.record"

    records = load_jsonl(manifest_path)

    count = 0
    with tf.io.TFRecordWriter(str(output_path)) as writer:
        for record in records:
            example = create_tf_example(record)
            writer.write(example.SerializeToString())
            count += 1

    print(f"{split_name}: {count} examples -> {output_path}")
    return output_path

train_record = write_tfrecord("train")
validation_record = write_tfrecord("validation")
test_record = write_tfrecord("test")

train: 1648 examples -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/train.record
validation: 354 examples -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/validation.record
test: 355 examples -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/test.record


## 4. Verify TFRecords

Read back a few records to confirm that serialization succeeded.

In [9]:
for record_path in [train_record, validation_record, test_record]:
    count = sum(1 for _ in tf.data.TFRecordDataset(str(record_path)))
    print(record_path.name, "records:", count)

train.record records: 1648
validation.record records: 354
test.record records: 355


## 5. Training Configuration Handoff

The following paths are the values Notebook 05 will use:

- train.record
- validation.record
- test.record
- configs/label_map.pbtxt
- configs/pipeline.config

**Application-side confirmation:** the web application does not consume TFRecords. TFRecords are training artifacts. The application will eventually consume the exported model from Notebook 07.